In [29]:
from dataclasses import dataclass, field
import pandas as pd
from typing import Callable
from collections import defaultdict

@dataclass
class InputData():
    """Example holder for inputs to a uniform step function."""
    
    names: list[str]

    favorite_color: dict[str, str]


@dataclass
class MeasurementData():
    """Holder for the successfull output of an example process."""

    data: pd.DataFrame

    title: str = field(default='')

    description: str = field(default='')


class ExampleProcessException(Exception):
    """Holder for errors related to the Example Process"""



@dataclass
class StepResult():
    """Holder that wraps successful results and/or errors from execution of
    one step of a sequence of uniform steps that comprise a Process.  
    
    The purpose of this wrapped result class is to provide a uniform method of
    handling success or failure cases."""

    step_name: str
    """Name of the step that produced this result"""

    value: MeasurementData | None = field(default=None)
    """Holds the successful output of a step function"""

    error: ExampleProcessException | None = field(default=None)
    """Holds an error if produced by calling a step function"""

    def __post_init__(self):
        # Make sure either value or error is set
        if self.value is None and self.error is None:
            message = "Either error or value or error must be set on a StepResult"
            message += f" : {self.step_name}"
            raise (ExampleProcessException(message))

    def is_ok(self) -> bool:
        return self.error is None and self.value is not None

# Type definition for a step function that takes InputData as a parameter
# and returns an instance of MeasurementData
StepFunction = Callable[[InputData], MeasurementData]

@dataclass
class BasicStep():

    step_name: str


    step_function: StepFunction


    def run(self, input_data: InputData) -> StepResult:
        """Calls the function assigned to this step and returns a StepResult with
        either the data or an error."""
        try:
            measurement_data = self.step_function(input_data)

            if measurement_data is None:
                # The step function returned None
                message = f"The function for {self.step_name} returned None"
                raise ExampleProcessException(message)
            
            elif not isinstance(measurement_data, MeasurementData):
                # Step function returned something that is not a MeasurementData object
                message = f"The function for {self.step_name} returned "
                message += "something other than a MeasurementData instance: "
                message += repr(measurement_data)
                raise ExampleProcessException(message)                
            
            else:
                # Everything seems to be ok
                return StepResult(step_name=self.step_name, value=measurement_data)
        
        except ExampleProcessException as epex:
                # There was an error
                return StepResult(step_name=self.step_name, error=epex)
            
        except Exception as ex:
                # Wrap the exception with the exception type our StepResult can hold
                message = f"Step {self.step_name} produced an error: {repr(ex)}"
                wrapped_exception = ExampleProcessException(message, ex)
                return StepResult(step_name=self.step_name, error=wrapped_exception)


@dataclass
class ProcessResult:

    step_outputs: dict[str, MeasurementData] = field(default_factory=dict)

    errors: dict[str, ExampleProcessException] = field(default_factory=dict)

    def add_result(self, step_result: StepResult) -> None:
        if step_result.is_ok():
            self.step_outputs[step_result.step_name] = step_result.value # type: ignore
        else:
            self.errors[step_result.step_name] = step_result.error # type: ignore

@dataclass
class Process:
    """A collection of steps and a run function that will call each step in
    sequence and collect the results into a ProcesssResult"""

    steps: list[BasicStep] = field(default_factory=list)


    def run(self, input_data: InputData) -> ProcessResult:

        process_result = ProcessResult()

        for step in self.steps:
            try:
                step_result = step.run(input_data)
                process_result.add_result(step_result)

            except ExampleProcessException as ex:
                process_result.add_result(StepResult(step.step_name, error=ex))

        # The process result should have 1 value or error for each step run
        assert len(self.steps) == len(process_result.step_outputs) + len(process_result.errors)

        return process_result


    @classmethod
    def create(cls, step_dict: dict[str, StepFunction]) -> 'Process':
        """Convenience function that creates a new process instance from a dict
         object mapping step names to step functions."""
        steps_list = []

        for step_name, step_function in step_dict.items():
            steps_list.append(BasicStep(step_name=step_name, step_function=step_function))
        
        return cls(steps=steps_list)


# Application of the pattern


# input data
###################################################################
family_names = ['Julie', 'James', 'Patrick']

our_favorite_colors = {
    'Julie': 'green',
    'James': 'green',
    'Patrick': 'blue'
}

our_input_data = InputData(names=family_names, favorite_color=our_favorite_colors)


# functions
###################################################################

def count_inputs(input_data: InputData) -> MeasurementData:
    raw_data = [{
        'name_count': len(input_data.names),
        'color_count': len(input_data.favorite_color.values())
        }]
    return MeasurementData(title="Name Count", data=pd.DataFrame(raw_data))


def count_by_first_letter(input_data: InputData) -> MeasurementData:
    raw_data = [{
        'j_names': len([n for n in input_data.names if n.startswith('J')]),
        'p_names': len([n for n in input_data.names if n.startswith('P')])
        }]
    return MeasurementData(title="Name Count", data=pd.DataFrame(raw_data))


def bad_step_function(input_data: InputData) -> MeasurementData:
    raise ValueError("A deliberate exeption!")


# Create the process
###################################################################

process = Process.create({
    "Count Inputs": count_inputs,
    "Count by First Letter": count_by_first_letter,
    "Demonstrate Error Handling": bad_step_function
})

process_result = process.run(our_input_data)


display("Successful Steps")
for step_name, measurement_data in process_result.step_outputs.items():
    display(step_name)
    display(measurement_data.data)

display("Errors")
for step_name, error in process_result.errors.items():
    display(step_name)
    display(error)



'Successful Steps'

'Count Inputs'

,name_count,color_count
0,3,3


'Count by First Letter'

,j_names,p_names
0,2,1


'Errors'

'Demonstrate Error Handling'

__main__.ExampleProcessException("Step Demonstrate Error Handling produced an error: ValueError('A deliberate exeption!')",
                                 ValueError('A deliberate exeption!'))